#### nn.RNN(GRU, LSTM도 동일)
```python
nn.RNN(input_size, hidden_size, batch_firt=True)
```
- input_size = 입력의 크기 = 각 단어의 벡터 표현의 차원 수
- hidden_size = 출력의 크기(output_dim)

In [ ]:
!pip install konlpy
!pip install mecab-python
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)


In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import urllib.request
from konlpy.tag import Mecab
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="ratings_test.txt")


In [ ]:
train_data = pd.read_table('ratings_train.txt')
test_data = pd.read_table('ratings_test.txt')

In [ ]:
print('훈련용 리뷰 개수: ', len(train_data))

In [ ]:
train_data[:5]

#### 2) 데이터 전처리

In [ ]:
# nunique(): 중복을 제거한 개수(ex: label은 0,1 2개)
train_data['document'].nunique(), train_data['label'].nunique()

In [ ]:
# document열의 중복 제거
# df.drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)
# subset: 중복 검사 열
# keep: 중복 제거 후 남길 행(first는 첫 값, last는 마지막 값)
# inplace: 원본 변경 여부
# ignore_index: 원래 index를 무시할지 여부
train_data.drop_duplicates(subset=['document'], inplace=True)

In [ ]:
train_data['label'].value_counts().plot(kind='bar')

In [ ]:
print(train_data.groupby('label').size().reset_index(name='count'))

In [ ]:
print(train_data.isnull().sum())

In [ ]:
train_data.loc[train_data.document.isnull()]

In [ ]:
# Null값 제거
train_data = train_data.dropna(how='any')

In [ ]:
len(train_data)

In [ ]:
# 한글과 공백 제외 모두 제거
# 온점과 같은 구두점 제거
train_data['document'] = train_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", regex=True)
train_data[:5]

In [ ]:
# 만약 한글이 포함되지 않은 document는 빈칸으로 됐을 수 있음
# 띄어쓰기만 되어있는 행을 empty value로 변경
train_data['document'] = train_data['document'].str.replace('^ +', "", regex=True)
# 공백을 Null값으로 변경
train_data['document'].replace('', np.nan, inplace=True)
print(train_data.isnull().sum())

In [ ]:
train_data = train_data.dropna(how='any')
len(train_data)

In [ ]:
# train_data전처리 과정과 같이 test_data도 진행
test_data.drop_duplicates(subset=['document'], inplace=True)
test_data['document'] = test_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", regex=True)
test_data['document'] = test_data['document'].str.replace('^ +', "", regex=True)
test_data['document'].replace('', np.nan, inplace=True)
test_data = test_data.dropna(how='any')
len(test_data)

#### 3) 토큰화

In [ ]:
# 불용어 지정
stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고', '을', '를', '인', '듯', '과', '와', '네', '들', '듯', '지', '임', '게']

In [ ]:
mecab = Mecab()

In [ ]:
mecab.morphs('와 이런 것도 영화라고 차라리 뮤직비디오를 만드는 게 나을 뻔')

In [ ]:
X_train = []
for sentence in tqdm(train_data['document']):
    tokenized_sentence = mecab.morphs(sentence)
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords]
    X_train.append(stopwords_removed_sentence)

In [ ]:
X_train[:3]

In [ ]:
X_test = []
for sentence in tqdm(test_data['document']):
    tokenized_sentence = mecab.morphs(sentence)
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords]
    X_test.append(stopwords_removed_sentence)

#### 4) 학습 데이터, 검증 데이터, 테스트 데이터

In [ ]:
y_train = np.array(train_data['label'])
y_test = np.array(test_data['label'])

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=0, stratify=y_train)

In [ ]:
print('--------학습 데이터의 비율-----------')
print(f'부정 리뷰 = {round(np.sum(y_train==0)/len(y_train) * 100,3)}%')
print(f'긍정 리뷰 = {round(np.count_nonzero(y_train)/len(y_train) * 100,3)}%')
print('--------검증 데이터의 비율-----------')
print(f'부정 리뷰 = {round(np.sum(y_valid==0)/len(y_valid) * 100,3)}%')
print(f'긍정 리뷰 = {round(np.count_nonzero(y_valid)/len(y_valid) * 100,3)}%')
print('--------테스트 데이터의 비율-----------')
print(f'부정 리뷰 = {round(np.sum(y_test==0)/len(y_test) * 100,3)}%')
print(f'긍정 리뷰 = {round(np.count_nonzero(y_test)/len(y_test) * 100,3)}%')


#### 5) 단어 집합 만들기

In [ ]:
word_list = []
for sent in X_train:
    for word in sent:
        word_list.append(word)

word_counts = Counter(word_list)

In [ ]:
print('총 단어수 :', len(word_counts))

In [ ]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print(vocab[:10])

In [ ]:
threshold_2 = 2
rare_cnt_2 = 0
rare_freq_2 = 0

threshold_3 = 3
rare_cnt_3 = 0
rare_freq_3 = 0

threshold_4 = 4
rare_cnt_4 = 0
rare_freq_4 = 0

total_cnt = len(word_counts)

total_freq = 0


for key, value in word_counts.items():
    total_freq = total_freq + value

    if(value < threshold_2):
        rare_cnt_2 = rare_cnt_2 + 1
        rare_freq_2 = rare_freq_2 + value

    if(value < threshold_3):
        rare_cnt_3 = rare_cnt_3 + 1
        rare_freq_3 = rare_freq_3 + value

    if(value < threshold_4):
        rare_cnt_4 = rare_cnt_4 + 1
        rare_freq_4 = rare_freq_4 + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold_2 - 1, rare_cnt_2))
print("단어 집합에서 3번 미만희귀 단어의 비율:", (rare_cnt_2 / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 3번 미만 등장 빈도 비율:", (rare_freq_2 / total_freq)*100)
print("-"*20)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold_3 - 1, rare_cnt_3))
print("단어 집합에서 3번 미만희귀 단어의 비율:", (rare_cnt_3 / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 3번 미만 등장 빈도 비율:", (rare_freq_3 / total_freq)*100)
print("-"*20)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold_4 - 1, rare_cnt_4))
print("단어 집합에서 3번 미만희귀 단어의 비율:", (rare_cnt_4 / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 3번 미만 등장 빈도 비율:", (rare_freq_4 / total_freq)*100)

In [ ]:
# 전체 단어 개수 중 빈도수 2 이하인 단어는 제거
vocab_size = total_cnt - rare_cnt_3
vocab = vocab[:vocab_size]
print(len(vocab))

In [ ]:
word_to_index = {}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

for index, word in enumerate(vocab):
    word_to_index[word] = index + 2

In [ ]:
print('단어 <PAD>와 맵핑되는 정수 :', word_to_index['<PAD>'])
print('단어 <UNK>와 맵핑되는 정수 :', word_to_index['<UNK>'])
print('단어 영화와 맵핑되는 정수 :', word_to_index['영화'])

In [ ]:
vocab_size = len(word_to_index)

#### 6. 정수 인코딩

In [ ]:
def texts_to_sequences(tokenized_X_data, word_to_index):
    encoded_X_data = []
    for sent in tokenized_X_data:
        index_sequences = []
        for word in sent:
            try:
                index_sequences.append(word_to_index[word])
            except KeyError:
                index_sequences.append(word_to_index['<UNK>'])
        encoded_X_data.append(index_sequences)
    return encoded_X_data

In [ ]:
encoded_X_train = texts_to_sequences(X_train, word_to_index)
encoded_X_valid = texts_to_sequences(X_valid, word_to_index)
encoded_X_test = texts_to_sequences(X_test, word_to_index)

In [ ]:
index_to_word = {}
for key, value in word_to_index.items():
    index_to_word[value] = key

#### 7. 패딩

In [ ]:
print('리뷰의 최대 길이: ', max(len(review) for review in encoded_X_train))
print('리뷰의 평균 길이: ', sum(map(len, encoded_X_train))/len(encoded_X_train))

plt.hist([len(review) for review in encoded_X_train], bins=50)
plt.show()

In [ ]:
def below_threshold_len(max_len, nested_list):
    count = 0
    for sentence in nested_list:
        if(len(sentence) <= max_len):
            count = count + 1
    print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'%(max_len, (count / len(nested_list))*100))

In [ ]:
# 그래프 상 30이 적당
max_len = 30
below_threshold_len(max_len, X_train)

In [ ]:
# 패딩 함수
def pad_sequences(sentences, max_len):
    features = np.zeros((len(sentences), max_len), dtype=int)
    for index, sentence in enumerate(sentences):
        if len(sentence) != 0:
            features[index,:len(sentence)] = np.array(sentence)[:max_len]
    return features

padded_X_train = pad_sequences(encoded_X_train, max_len=max_len)
padded_X_valid = pad_sequences(encoded_X_valid, max_len=max_len)
padded_X_test = pad_sequences(encoded_X_test, max_len=max_len)

print('훈련 데이터의 크기 :', padded_X_train.shape)
print('검증 데이터의 크기 :', padded_X_valid.shape)
print('테스트 데이터의 크기 :', padded_X_test.shape)

In [ ]:
print('첫번째 샘플의 길이 :', len(padded_X_train[0]))
print('첫번째 샘플 :', padded_X_train[0])

## LSTM을 이용한 네이버 영화 리뷰 분류 모델

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cup")
print("cpu와 cuda 중 다음 기기로 학습함:", device)

In [ ]:
train_label_tensor = torch.tensor(np.array(y_train))
valid_label_tensor = torch.tensor(np.array(y_valid))
test_label_tensor = torch.tensor(np.array(y_test))

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_length)
        # (batch_size, seq_length, embedding_dim)
        embedded = self.embedding(x)
        # LSTM은 (hidden state, cell state)의 튜플을 반환한다.
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # lstem_out: (batch_size, seq_length, hidden_dim), hidden: (1, batch_size, hidden_dim)
        last_hidden = hidden.squeeze(0) # (batch_size, hidden_dim)
        logits = self.fc(last_hidden)
        return logits

In [ ]:
encoded_train = torch.tensor(padded_X_train).to(torch.int64)
train_dataset = torch.utils.data.TensorDataset(encoded_train, train_label_tensor)
train_dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=32)

encoded_test = torch.tensor(padded_X_test).to(torch.int64)
test_dataset = torch.utils.data.TensorDataset(encoded_test, test_label_tensor)
test_dataloader = torch.utils.data.DataLoader(test_dataset, shuffle=True, batch_size=1)

encoded_valid = torch.tensor(padded_X_valid).to(torch.int64)
valid_dataset = torch.utils.data.TensorDataset(encoded_valid, valid_label_tensor)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, shuffle=True, batch_size=1)

In [ ]:
total_batch = len(train_dataloader)
total_batch

In [ ]:
embedding_dim = 100
hidden_dim = 128
output_dim = 2 #0,1로 분류
learning_rate = 0.01
num_epochs = 10

model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
model.to(device)

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
## 평가 코드
def calculate_accuracy(logits, labels):
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted==labels).sum().item()
    total = labels.size(0)
    accuracy = correct/total
    return accuracy

In [ ]:
def evaluate(model, valid_dataloader, loss_function, device):
    val_loss = 0
    val_correct = 0
    val_total = 0

    #모델을 평가 모드로 설정
    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in valid_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            logits = model(batch_X)
            loss = loss_function(logits, batch_y)

            val_loss += loss.item()
            val_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
            val_total += batch_y.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= len(valid_dataloader)

    return val_loss, val_accuracy

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss = 0
    train_correct = 0
    train_total = 0
    model.train()

    for batch_X, batch_y in tqdm(train_dataloader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        logits = model(batch_X)

        loss = loss_function(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
        train_total += batch_y.size(0)

    train_accuracy = train_correct/ train_total
    train_loss /= len(train_dataloader)

    val_loss, val_accuracy = evaluate(model, valid_dataloader, loss_function, device)

    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

    # 검증 손실이 최소일 때 체크포인트 저장
    if val_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. 체크포인트를 저장합니다.')
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_checkpoint.pth')

In [ ]:
# 모델 검증하기
model.load_state_dict(torch.load('best_model_checkpoint.pth'))
model.to(device)

In [ ]:
val_loss, val_accuracy = evaluate(model, valid_dataloader, loss_function, device)

print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

In [ ]:
test_loss, test_accuracy = evaluate(model, test_dataloader, loss_function, device)

print(f'Best model test loss: {test_loss:.4f}')
print(f'Best model test accuracy: {test_accuracy:.4f}')

In [ ]:
# 모델 테스트
index_to_tag = {0:"부정", 1:"긍정"}

def predict(text, model, word_to_index, index_to_tag):
    model.eval()

    tokens = mecab.morphs(text)
    tokens = [word for word in tokens if not word in stopwords]
    token_indices = [word_to_index.get(token, 1) for token in tokens]

    input_tensor = torch.tensor([token_indices], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_tensor)

    predicted_index = torch.argmax(logits, dim=1)
    predicted_tag = index_to_tag[predicted_index.item()]

    return predicted_tag


In [ ]:
text = "이 영화 개꿀잼 ㅋㅋ"
predict(text, model, word_to_index, index_to_tag)

In [ ]:
text = "감독 뭐하는 놈이냐?"
predict(text, model, word_to_index, index_to_tag)

In [ ]:
text = "어떻게 이 영화를 아직 안봤어?"
predict(text, model, word_to_index, index_to_tag)